# 2 · Surrogate modelling

**What you will do.** Simulate a set of designs, train statistical models on the results, compare them on data they have not seen, and keep the best one for the optimization.

**You need.** Notebook 1 (the vocabulary: parameter, problem, evaluator, sampling). The simulated samples of the second part are stored in `surrogate_simple_simulation.csv`, so you do not have to wait for them.

**Run time.** about 3 minutes with the stored simulations. With `RESIMULATE = True` (below) it runs the 50 simulations of part 2 itself: about 16 minutes in all.

The four notebooks of this series use the same building model (`test.idf`, a small house) and the same weather file
(Turin). Work in order: 1 *bdopt overview* → 2 *surrogate modelling* → 3 *optimization* → 4 *adaptive sampling*. Cells
marked **Your turn** are exercises: change something and observe what happens.

## Why a surrogate?

Surrogate modelling or metamodelling is a way to emulate physics-based building simulation models [[1]]. A one-year
EnergyPlus simulation takes tens of seconds to minutes; a design study or an optimisation needs thousands of
evaluations. A surrogate is trained on a few hundred simulations and then answers in milliseconds.

bdopt integrates building simulation and surrogate modelling using:
+ EnergyPlus model editing and execution via evaluators (`EnergyPlusEvaluator`)
+ sampling (`bdopt.sampling`, built on [scipy](https://docs.scipy.org/doc/scipy/reference/stats.qmc.html))
+ machine learning with [scikit-learn](https://scikit-learn.org/stable/) (`bdopt.surrogate`)

[1]: https://doi.org/10.1016/j.enbuild.2019.05.057 (Westermann, P., Evins, R. (2019). Surrogate modelling for sustainable building design - a review. Energy and Buildings 198, 170-186.)

## The workflow

The figure shows the workflow of a surrogate-based design study and, next to the boxes, the parts of bdopt that
do each step. This notebook covers the first four boxes: the problem (notebook 1), the samples, the surrogate that is
fitted to them, and its validation. The last box, design optimization, is notebook 3.

The dashed arrows say that the study is not a straight line: if the surrogate is not accurate enough, or the optimiser
needs more accuracy where it is looking, you go back and simulate more designs. The grey boxes are other uses of a
validated surrogate, which these notebooks do not develop.

![The workflow of a surrogate-based design study](image/surrogate_overview.png)

*Figure from the course notebooks, which were written for BESOS; the names of the toolbox were corrected.*

### The steps

+ define a problem consisting of parameters to vary with ranges and an objective function
+ sample the problem space
+ define an evaluator by linking the problem to an EnergyPlus model
+ run the evaluator for the samples
+ train a surrogate model over the samples
+ explore the design space using an interactive plot that queries the surrogate model

In [ ]:
%matplotlib inline
import time
from pathlib import Path

import numpy as np
import pandas as pd
from ipywidgets import FloatSlider, interact
from matplotlib import pyplot as plt
from sklearn import linear_model, pipeline
from sklearn.preprocessing import StandardScaler

from bdopt import (EnergyPlus, EnergyPlusEvaluator, FieldSelector, FunctionEvaluator, IntegerParameter,
                   Parameter, Problem, RangeParameter, load_idf, sampling, viz)
from window_to_wall import wwr      # the window-to-wall ratio as a design variable: see 1_bdopt_overview.ipynb

N_SAMPLES = 50          # simulations of the second part of the notebook
RESIMULATE = False      # False: use the stored simulations (surrogate_simple_simulation.csv); True: run them yourself (~10 min)

# Part 1 · A first surrogate, with two variables

## Building model
We load an EnergyPlus model.

In [ ]:
idf = load_idf('test.idf')
epw_file = 'ITA_Torino.160590_IWEC.epw'
idf

## Parameters and Objectives
We start by defining `Parameters` that govern how we would like to modify the building.
Here they are the thickness of the insulation layer and the window-to-wall ratio. The first is a field of a material,
written with a `FieldSelector`; the second is a geometry edit, written as a `GenericSelector` in `window_to_wall.py`
(see `1_bdopt_overview.ipynb`).

In [ ]:
parameters = [
    Parameter(FieldSelector('Material', 'XPS Extruded Polystyrene  - CO2 Blowing_0041', 'Thickness'),
              RangeParameter(0.001, 0.3), name='Insulation Thickness'),      # [m]
    wwr(0.1, 0.8),                                                           # window-to-wall ratio between 0.1 and 0.8
]

Now, we specify that we would like to optimize for the heating use of the entire facility. We then bundle all of this information together.

In [ ]:
objectives = ["DistrictHeating:Facility"]
problem = Problem(parameters, objectives)

## Sampling

We need samples on which to train a surrogate model. The sampler uses Latin Hypercube Sampling: each variable's range
is cut in as many slices as there are samples and every slice is used once, so the designs are spread over the whole
design space. A `seed` makes the choice repeatable. We can check how well distributed they are by looking at the table.

In [ ]:
inputs = sampling.lhs(problem, 10, seed=0)
inputs

## Evaluator
We now define and run the Evaluator, which will run the EnergyPlus instances with the different inputs generated in the previous cell.

In [ ]:
evaluator = EnergyPlusEvaluator(problem, idf, epw_file,
                                out_dir='/tmp/bdopt_runs',   # work files go to the container's own disk (the shared folder is much slower)
                                err_dir='outputdir',         # runs that fail are kept here, with the EnergyPlus error file
                                processes=2)                 # EnergyPlus runs in parallel
t1 = time.time()
train = evaluator.evaluate(inputs, keep_input=True)       # parameters + objective, one row per simulation

t2 = time.time()
train.head()  # first 5 lines

In [ ]:
def niceformat(seconds):
    seconds = seconds % (24 * 3600)
    hour = seconds // 3600
    seconds %= 3600
    minutes = seconds // 60
    seconds %= 60
    return hour, minutes, seconds

In [ ]:
time_of_sim = t2 - t1
hours, mins, secs = niceformat(time_of_sim)

print(
    "The total running time: {:2.0f} hours {:2.0f} min {:2.0f} seconds".format(
        hours, mins, secs
    )
)

We can save results at this point:

In [ ]:
train.to_pickle("samples.p")

And load them again later:

In [ ]:
train = pd.read_pickle("samples.p")

## Exploring the training data
Let's look at the relation between the inputs and the output with a bdopt plot: one panel per objective and per parameter.

In [ ]:
viz.pairplot(train, problem, size=3)
plt.show()

**Your turn.** Does more insulation always lower the heating demand? What does the plot say about the window-to-wall ratio?
With only 10 samples, could you be fooled? Keep this question in mind for the second part.

Here's a plot of the _design space_ defined by the samples.

In [ ]:
x, y, obj = problem.names()
train.plot.hexbin(x=x, y=y, C=obj, cmap="plasma", sharex=False, gridsize=50)

In [ ]:
problem.names()

## Fitting a surrogate model

Now we can train the model using a scikit-learn pipeline:

In [ ]:
train

In [ ]:
model = pipeline.make_pipeline(StandardScaler(), linear_model.LinearRegression())
model.fit(train[[x, y]].values, train[obj].values)
print('R2 on the training data:', round(model.score(train[[x, y]].values, train[obj].values), 3))

## Running the surrogate model

We can query the model at a specific point in the design space (insulation thickness 0.2 m, window-to-wall ratio 0.25):

In [ ]:
model.predict([[0.2, 0.25]])

Or for a `dataframe` made from a `dictionary` of points:

In [ ]:
points = pd.DataFrame(data={x: [0.05, 0.25], y: [0.2, 0.6]})
model.predict(points.values)

Next we define a function for querying the surrogate model across the domain at a given density, and plot the result.

In [ ]:
def run_model(model, density):
    p1 = problem.parameters[0].value_descriptor
    a = np.linspace(p1.min, p1.max, density)
    p2 = problem.parameters[1].value_descriptor
    b = np.linspace(p2.min, p2.max, density)
    plot_data = pd.DataFrame(
        np.transpose([np.tile(a, len(b)), np.repeat(b, len(a))]),
        columns=problem.parameter_names,
    )
    return pd.concat([plot_data, pd.Series(model.predict(plot_data.values))], axis=1)


density = 200
df = run_model(model, density)
df.plot.hexbin(x=x, y=y, C=0, cmap="plasma", gridsize=density, sharex=False)

## Interactive surrogate
We make an interactive plot using `ipywidgets`, which executes the model every time a slider is moved.

In [ ]:
# get the min and max of the two variables
(min1, max1), (min2, max2) = (
    (float(p.value_descriptor.min), float(p.value_descriptor.max)) for p in problem.parameters
)
# get the data to plot in the background
df = run_model(model, 100)

# define a wrapper to be queried by interact
def model_wrapper(v1, v2):
    df.plot.hexbin(
        x=x, y=y, C=0, cmap="plasma", gridsize=30, sharex=False
    )  # background plot
    x_lims = plt.xlim()
    y_lims = plt.ylim()
    plt.scatter(x=v1, y=v2)  # plot the marker
    plt.xlim((x_lims))
    plt.ylim((y_lims))
    plt.show()

    value = (
        model.predict([[v1, v2]])[0] / 3.6e6
    )  # find the current value and convert to kWh
    output = (
        "Heating use = " + str(value.round()) + " kWh"
    )  # return the string to display
    return output


# make the interactive plot with two sliders
continuous_update = False
interact(
    model_wrapper,
    v1=FloatSlider(
        min=min1,
        max=max1,
        value=(min1 + max1) / 2,
        description="Thickness [m]",
        step=0.01,
        continuous_update=continuous_update,
    ),
    v2=FloatSlider(
        min=min2,
        max=max2,
        value=(min2 + max2) / 2,
        description="WWR",
        step=0.01,
        continuous_update=continuous_update,
    ),
)

# Part 2 · Gaussian process vs neural networks

Let's compare now a Gaussian process with a neural network and a deeper network for surrogate model creation, on a more
complex design space: five design variables and three objectives, trained on **50** simulated designs.

If a model fits the 10 designs it was trained on, that proves nothing: it may have memorised them. What matters is the
error on designs it has *never seen*, so we keep part of the simulations aside as a test set.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning
import warnings

warnings.filterwarnings('ignore', category=ConvergenceWarning)   # the networks do not always "converge" on a handful of samples

In [ ]:
insulation = FieldSelector(class_name='Material', object_name='XPS Extruded Polystyrene  - CO2 Blowing_0041', field_name='Thickness')
insulation_range = RangeParameter(min_val=0.01, max_val=0.45)
insulation_param = Parameter(selector=insulation,
                             value_descriptor=insulation_range,
                             name='Insulation Thickness')

In [ ]:
lights_selector = FieldSelector(class_name='Lights', object_name='*', field_name='Watts per Zone Floor Area')
single_digit_integers = IntegerParameter(8, 12)         # whole numbers of W/m2, 8 to 12 included
lights_param = Parameter(
    lights_selector,
    value_descriptor=single_digit_integers,
    name="Lights Watts/Area",
)

In [ ]:
# one parameter per field of the glazing material named "3"
more_parameters = [
    Parameter(FieldSelector('WindowMaterial:Glazing', '3', 'Conductivity'),
              RangeParameter(0.1, 1), name='Glazing conductivity'),
    Parameter(FieldSelector('WindowMaterial:Glazing', '3', 'Solar Transmittance at Normal Incidence'),
              RangeParameter(0.01, 0.8), name='Glazing solar transmittance'),
]

In [ ]:
window_to_wall = wwr(0.1, 0.8)

In [ ]:
parameters = [insulation_param] + more_parameters + [window_to_wall] + [lights_param]

In [ ]:
objectives = ['Electricity:Facility', 'DistrictHeating:Facility', 'DistrictCooling:Facility']   # EnergyPlus meters, summed over the run period [J]

problem = Problem(parameters, objectives)    # problem = parameters + objectives

In [ ]:
samples = sampling.lhs(problem, N_SAMPLES, seed=0)
samples.head()

In [ ]:
evaluator = EnergyPlusEvaluator(problem, idf, epw_file,
                                out_dir='/tmp/bdopt_runs',   # work files go to the container's own disk (the shared folder is much slower)
                                err_dir='outputdir',         # runs that fail are kept here, with the EnergyPlus error file
                                processes=2)

saved = Path('surrogate_simple_simulation.csv')
sim_samples = None
if saved.exists() and not RESIMULATE:
    table = pd.read_csv(saved, index_col=0)
    if list(table.columns) == problem.names():          # a file written for another problem is ignored
        sim_samples = table
        print(f'Using the {len(sim_samples)} simulations stored in {saved}.')
if sim_samples is None:
    print(f'Simulating {len(samples)} designs (about 12 minutes)...')
    t1 = time.time()
    sim_samples = evaluator.evaluate(samples, keep_input=True)
    hours, mins, secs = niceformat(time.time() - t1)
    print("The total running time: {:2.0f} hours {:2.0f} min {:2.0f} seconds".format(hours, mins, secs))
    sim_samples.to_csv(saved)

In [ ]:
sim_samples.columns

In [ ]:
features = list(samples.columns)
print(features)
print(objectives)

In [ ]:
# remove inputs
outputs = sim_samples.drop(features, axis=1)
# outputs

In [ ]:
# remove outputs
inputs = sim_samples.drop(objectives, axis=1)
# inputs

### Which variables matter?

Before training anything, look at how each objective responds to each variable (Pearson correlation, from -1 to 1). Everything a surrogate has to learn is in this table.

In [ ]:
sim_samples.corr().loc[objectives, features].round(2)

In this model the electricity consumption depends on the lighting power alone (correlation 1.00): there is nothing to learn there, and the models will get that objective right. Heating and cooling are driven mostly by the window-to-wall ratio and by the glazing: that is where the models differ.

## Train-test split

Next we split the data into a training set (80%, 40 designs) and a testing set (20%, 10 designs). The test designs are
never used to fit anything: they only measure how the models behave on new designs.

In [ ]:
train_in, test_in, train_out, test_out = train_test_split(
    inputs, outputs, test_size=0.2, random_state=0
)

## Normalization of inputs

To ensure an equal weighting of inputs and outputs in the training of the networks, we normalise the values.
For example the window-to-wall ratio is in the range of 0 to 1 while the lighting power is in a range of 8 to 12 W/m$^2$.
Different options for normalisation exist. Here we bring every variable to zero mean and standard deviation 1.

**Fit the normaliser on the training data only, and only *apply* it to the test data.** If the test data took part in
computing the mean and standard deviation, information about the test set would leak into the model and the test error
would look better than it really is.

In [ ]:
scaler = StandardScaler()
train_in_scale = scaler.fit_transform(X=train_in)
test_in_scale = scaler.transform(X=test_in)

scaler_out = StandardScaler()
train_out_scale = scaler_out.fit_transform(X=train_out)

# Gaussian Process


## Hyper-parameters

Before fitting the GP model we define the set of hyperparameters we want to optimize.
Here we use 3 folds in the k-fold cross validation scheme.
We select a set of Kernel functions, which must fit the characteristics of a problem - details and examples may be found in the [Kernel cookbook](https://www.cs.toronto.edu/~duvenaud/cookbook/).
Note that the parameters of the Kernel itself are [optimized during each model fitting run](https://scikit-learn.org/stable/modules/generated/sklearn.gaussian_process.GaussianProcessRegressor.html).

In [ ]:
hyperparameters = {
    "kernel": [
        None,
        1.0 * RBF(length_scale=1.0, length_scale_bounds=(1e-1, 10.0)),
        1.0 * RationalQuadratic(length_scale=1.0, alpha=0.5),
        1.0 * Matern(length_scale=1.0, length_scale_bounds=(1e-1, 10.0)),
    ]
}

folds = 3

## Model fitting

Here we fit the model using these hyperparameters.

In [ ]:
gp = GaussianProcessRegressor(normalize_y=True)

clf = GridSearchCV(gp, hyperparameters, cv=folds)

clf.fit(train_in_scale, train_out_scale)

best_gp = clf.best_estimator_

In [ ]:
prediction_gp = scaler_out.inverse_transform(best_gp.predict(test_in_scale))

# Neural Network

## Hyper-parameters

Before we start fitting the NN model we define the set of hyperparameters we want to analyse in our cross-validation to optimize the model.
Here, we select the number of layers of the network as well as the regularization parameter alpha as parameter value.
A larger number of layers and a lower value of the regularizer lead to higher variance of the network.
This may lead to overfitting.
The best selection may be found using an optimizer like Bayesian Optimization.
In this example we use a simple grid search.

In [ ]:
hyperparameters = {
    "hidden_layer_sizes": (
        (len(parameters) * 16,),
        (len(parameters) * 16, len(parameters) * 16),
    ),
    "alpha": [1, 10, 10 ** 3],
}

neural_net = MLPRegressor(max_iter=1000, early_stopping=False, random_state=0)
folds = 3

## Model fitting

Here, we use the NN model from scikit-learn.

In [ ]:
clf = GridSearchCV(neural_net, hyperparameters, cv=folds)
clf.fit(train_in_scale, train_out_scale)

nn_model = clf.best_estimator_

In [ ]:
prediction_nn = scaler_out.inverse_transform(nn_model.predict(test_in_scale))

## A deeper network

Three hidden layers (32, 64 and 32 neurons, ReLU), trained with the Adam optimizer for 1000 epochs. No deep-learning framework is installed in the course environment: scikit-learn's `MLPRegressor` trains the same architecture. (bdopt also has a `TorchMLP` surrogate, which needs PyTorch.)

In [ ]:
deep_model = MLPRegressor(hidden_layer_sizes=(32, 64, 32), activation='relu', solver='adam',
                          learning_rate_init=0.001, batch_size=min(32, len(train_in_scale)), max_iter=1000, random_state=0)
deep_model.fit(train_in_scale, train_out_scale)

In [ ]:
prediction_deep = scaler_out.inverse_transform(deep_model.predict(test_in_scale))

## Compare the models

RMSE on the test set, as a percentage of the mean of each objective (smaller is better).

In [ ]:
def rmse_percent(prediction):
    return root_mean_squared_error(test_out.values, prediction, multioutput='raw_values') / test_out.mean().values * 100


pd.DataFrame({'GP': rmse_percent(prediction_gp),
              'NN': rmse_percent(prediction_nn),
              'deep NN': rmse_percent(prediction_deep)}, index=objectives)

A model with an error of a few percent of the mean is usually good enough to rank designs. With 10 test designs the comparison is only indicative: change `random_state` in `train_test_split` and see whether the ranking of the models changes.

### The same models with `bdopt.surrogate`

`bdopt.surrogate` offers these models ready-made. They scale inputs and outputs inside, so the normalisation, the grid search and the split bookkeeping above disappear, and `score` gives the same percentage error.

In [ ]:
from bdopt.surrogate import GPSurrogate, MLPSurrogate

train_df = pd.concat([train_in, train_out], axis=1)
test_df = pd.concat([test_in, test_out], axis=1)

bdopt_models = {'GP': GPSurrogate(), 'NN': MLPSurrogate(random_state=0)}
for model_ in bdopt_models.values():
    model_.fit(train_df, problem)

pd.DataFrame({name: m.score(test_df)['rmse_pct'] for name, m in bdopt_models.items()})

In [ ]:
# predicted against simulated, on the test designs (points on the diagonal are perfect predictions)
viz.parity(test_out, bdopt_models['GP'].predict(test_in))
plt.show()

**Your turn.** Which surrogate would you use? Try `make_surrogate('random_forest')` (see `help(make_surrogate)` for the other names) and compare it in the same way.

## Surrogate Modelling Evaluator object
We can wrap the fitted model in a bdopt `FunctionEvaluator`: an evaluator that computes the objectives with a Python function instead of running EnergyPlus. It receives a table of designs, one column per parameter.

In [ ]:
# definition of the evaluation function
def evaluation_func(designs):
    return scaler_out.inverse_transform(nn_model.predict(scaler.transform(designs[features])))


surrogate_evaluator = FunctionEvaluator(problem, evaluation_func, vectorized=True)

In [ ]:
srinputs = sampling.lhs(problem, 500, seed=1)
sroutputs = surrogate_evaluator.evaluate(srinputs)
srresults = srinputs.join(sroutputs)
srresults.head()

## Exploration

A brushable parallel-coordinates plot, as in the original course notebooks: one line per design (500 designs answered by
the neural network), one axis per variable, coloured by the electricity use. **Drag on an axis** to keep only the designs
whose value lies in that range: select the lowest heating demand and see which values of the parameters go with it.
Drag the range to move it, click on the axis to remove it, drag the title of an axis to put it elsewhere.

In [ ]:
import explore       # explore.py, next to this notebook: the tools of the next part

explore.parallel_coordinates(srresults, problem, color='Electricity:Facility').show()

# Part 3 · Looking inside the surrogate

A surrogate is a function that answers in milliseconds, so you can ask it anything. Here are four ways of looking inside.
They are all in `explore.py` (open it from the Jupyter file list: it is plain Python). For this part we train a Gaussian
process on all 50 simulations, because it can also say **how sure it is** of each prediction.

In [ ]:
from bdopt.surrogate import GPSurrogate

gp = GPSurrogate().fit(sim_samples, problem)         # all 50 simulated designs
print('Can the model say how sure it is?', gp.has_uncertainty)

### 1 · Move the parameters

One slider per parameter. The tabs show, for the design you build: the **Prediction** (bars, with the model's uncertainty as
error bars, and the range of the simulated designs in grey), the **Sections** (each objective against each parameter, the
others kept where the sliders are), a **Map** of two parameters (switch to *uncertainty* to see where the model is unsure)
and a **Compare** of two saved designs.

In [ ]:
ex = explore.ParameterExplorer(gp, problem, training=sim_samples)
ex.widget

**Your turn.**
- Move the window-to-wall ratio. What happens to cooling? And to heating? Does it agree with the *Sections* tab?
- *Map*: x = window-to-wall ratio, y = glazing solar transmittance, objective = cooling. Then switch to *uncertainty*: where is the model least sure? (Far from the white dots, the simulated designs.) Where would you simulate next?
- Press *Save as A*, change two parameters, press *Save as B*: the *Compare* tab gives the difference in kWh and in %.
- `ex.design` is the design you have built, as a dictionary: you can hand it to `EnergyPlusEvaluator.evaluate` to check the prediction.

### 2 · Which parameters matter?

Two answers. The **tornado**: starting from the middle of every range, how far does each objective move when *only* one parameter goes from its minimum to its maximum? The **Sobol' indices**: the share of the variance of an objective that a parameter explains *alone* (S1) and *together with the others* (ST); an ST much larger than S1 means the parameter acts through an interaction. With the surrogate this takes seconds (about 14 000 evaluations): with EnergyPlus it would take days.

In [ ]:
explore.tornado(gp, problem)

In [ ]:
S1, ST = explore.sobol_indices(gp, problem)
explore.sensitivity_plot(S1, ST)
pd.concat({'S1': S1, 'ST': ST}, axis=1).round(2)

**Your turn.** Which parameter would you stop worrying about? Which two parameters interact (ST well above S1)? Compare with the correlation table you looked at before: what does a correlation miss?

### 3 · Does the surrogate need more simulations?

Train on 10, 20, 30, 40 designs, always testing on designs that were left out, with a few random choices of the training designs for each size. If the error keeps falling, more simulations would still help; if the curve is flat, they would not.

In [ ]:
explore.learning_curve(lambda: GPSurrogate(), sim_samples, problem, sizes=(10, 20, 30, 40), repeats=2)

**Your turn.** What would you do with a budget of 20 more simulations? (The *uncertainty* map points at where; the adaptive sampler of bdopt, `bdopt.adaptive`, does it automatically: notebook 4.)

## Save the surrogate

`3_optimization.ipynb` uses a surrogate stored in a file. bdopt surrogates scale their inputs and outputs inside, so the
file is all that is needed (no separate scalers). The one below is the neural network with its grid search, trained on
**all** the simulations (train and test: once the model is chosen, every simulation helps).

In [ ]:
from bdopt.surrogate import MLPSurrogate

best_surrogate = MLPSurrogate(random_state=0).fit(sim_samples, problem)
best_surrogate.save('best_surrogate.pkl')

## Summary
- Simulated samples are the expensive part; a **surrogate** trained on them answers instantly.
- Judge a model on **data it has not seen**; fit the normaliser on the training data only.
- With few samples the comparison between models is only indicative.

Next: **3_optimization.ipynb** uses the surrogate to search for the best designs.

## Sources and credits

**Origin of this series.** The four notebooks are adapted from the notebooks of the course, which were
written for the toolbox BESOS (Westermann, P., Christiaanse, T.V., Beckett, W., Kovacs, P., Evins, R. (2021). besos: Building and Energy Simulation, Optimization and Surrogate Modelling. *Journal of Open Source Software* 6(60), 2677. https://doi.org/10.21105/joss.02677; code and examples under GPL-3.0, https://gitlab.com/energyincities/besos). They were
rewritten in October 2026 for **bdopt** (MIT licence, https://github.com/LorenzoBottaccioli/bdopt) and extended; the figures marked
above come from those notebooks or from the BESOS documentation.

**Model and weather.** `test.idf` was generated with DesignBuilder for the course. `ITA_Torino.160590_IWEC.epw`: IWEC data,
original source data (c) 2001 American Society of Heating, Refrigerating and Air-Conditioning Engineers (ASHRAE), Inc., Atlanta,
GA, USA, converted to EPW for EnergyPlus; licensed for your own use, not to be passed to third parties without ASHRAE's permission.

**Simulation.** The building is simulated with EnergyPlus version 9.6 (U.S. Department of Energy and partners, BSD-style licence).

**References for this notebook.**
- Crawley, D.B., Lawrie, L.K., Winkelmann, F.C., Buhl, W.F., Huang, Y.J., Pedersen, C.O., Strand, R.K., Liesen, R.J., Fisher, D.E., Witte, M.J., Glazer, J. (2001). EnergyPlus: creating a new-generation building energy simulation program. *Energy and Buildings* 33(4), 319-331. https://doi.org/10.1016/S0378-7788(00)00114-6
- Westermann, P., Evins, R. (2019). Surrogate modelling for sustainable building design - a review. *Energy and Buildings* 198, 170-186. https://doi.org/10.1016/j.enbuild.2019.05.057
- Rasmussen, C.E., Williams, C.K.I. (2006). *Gaussian Processes for Machine Learning*. MIT Press. ISBN 0-262-18253-X. https://gaussianprocess.org/gpml/
- Pedregosa, F. et al. (2011). Scikit-learn: machine learning in Python. *Journal of Machine Learning Research* 12, 2825-2830. https://jmlr.org/papers/v12/pedregosa11a.html
- McKay, M.D., Beckman, R.J., Conover, W.J. (1979). A comparison of three methods for selecting values of input variables in the analysis of output from a computer code. *Technometrics* 21(2), 239-245. https://doi.org/10.1080/00401706.1979.10489755
- Sobol', I.M. (2001). Global sensitivity indices for nonlinear mathematical models and their Monte Carlo estimates. *Mathematics and Computers in Simulation* 55(1-3), 271-280. https://doi.org/10.1016/S0378-4754(00)00270-6

All of the platform's credits and licences are listed on the page **Credits & licences** of the platform.